In [1]:
import os
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langchain_core.agents import AgentActionMessageLog

# --- 1. Importujemy Twoją konfigurację modeli ---
from openrouter_free import FREE_MODELS, OpenRouterFree

# Sprawdzamy klucz (używamy Twojej klasy do weryfikacji)
try:
    _ = OpenRouterFree()
except ValueError as e:
    print(f"Błąd: {e}")
    # Wyjdź lub obsłuż brak klucza

# --- 2. Definiujemy narzędzie (tool) ---
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an order that hasn't shipped."""
    # Tutaj Twoja logika backendowa
    return f"Order {order_id} has been cancelled."

# --- 3. Konfiguracja Modelu (LangChain 0.3.7 + OpenRouter) ---
# Używamy ChatOpenAI, ale ustawiamy base_url na OpenRouter
# i wybieramy model z Twojego słownika FREE_MODELS
llm = ChatOpenAI(
    model=FREE_MODELS["auto"],  # Przykład: deepseek-r1:free
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
).bind_tools([cancel_order])  # Przypisujemy narzędzia

# --- 4. Logika Agenta (Dostosowana do LangGraph 0.3.7) ---
def call_model(state: dict):
    messages = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})
    
    # System prompt
    prompt = (
        f"You are an ecommerce support agent.\n"
        f"ORDER ID: {order['order_id']}\n"
        f"If the customer asks to cancel, call cancel_order(order_id) "
        f"and then send a simple confirmation. Otherwise, just respond normally."
    )
    
    # W LangChain 0.3+ używamy invoke na messages
    # Dodajemy SystemMessage na początek
    full_messages = [SystemMessage(content=prompt)] + messages
    
    response = llm.invoke(full_messages)
    
    # Jeśli model wywołał tool
    if hasattr(response, "tool_calls") and response.tool_calls:
        tool_call = response.tool_calls[0]
        if tool_call["name"] == "cancel_order":
            # Wykonaj narzędzie
            result = cancel_order(**tool_call["args"])
            # Zwróć odpowiedź z ToolMessage
            return {"messages": [response, ToolMessage(content=result, tool_call_id=tool_call["id"])]}
    
    return {"messages": [response]}

# --- 5. Budowa Grafu (LangGraph 0.3.7 style) ---
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    order: dict

def construct_graph():
    g = StateGraph(AgentState)
    g.add_node("assistant", call_model)
    g.add_edge(START, "assistant")
    g.add_edge("assistant", END)
    return g.compile()

# --- 6. Uruchomienie ---
if __name__ == "__main__":
    graph = construct_graph()
    
    example_order = {"order_id": "A12345"}
    convo = [HumanMessage(content="Please cancel my order A12345.")]
    
    # invoke w wersji 0.3.7
    result = graph.invoke({"order": example_order, "messages": convo})
    
    for msg in result["messages"]:
        print(f"{msg.type}: {msg.content}")


human: Please cancel my order A12345.
ai: OLCALL>[{"name": "cancel_order", "arguments": {"order_id": "A12345ALL>



TOOLS

In [3]:
import os
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain.tools import tool

# --- 1. Importujemy konfigurację modeli z Twojego openrouter.py ---
from openrouter_free import FREE_MODELS

# --- 2. Definicja narzędzi (tools) ---
@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' times 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the 'y'."""
    return x ** y

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

# --- 3. Inicjalizacja LLM z OpenRouterem i podpięcie narzędzi ---
tools = [multiply, exponentiate, add]

llm = ChatOpenAI(
    model=FREE_MODELS["auto"],  # Wybieramy model z Twojej listy
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

# "Binding" - rejestracja narzędzi przy modelu
llm_with_tools = llm.bind_tools(tools)

# --- 4. Wykonanie zapytania ---
query = "What is 393 * 12.25? Also, what is 11 + 49?"
messages = [HumanMessage(query)]

# Pierwsze wywołanie - model decyduje o użyciu narzędzi
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

# --- 5. Obsługa wywołań narzędzi ---
if hasattr(ai_msg, "tool_calls") and ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        # Mapowanie nazwy wywołania na funkcję
        tool_map = {
            "multiply": multiply,
            "exponentiate": exponentiate,
            "add": add,
        }
        
        # Pobieramy funkcję (LangChain 0.3.7 zaleca użycie .invoke)
        selected_tool = tool_map.get(tool_call["name"].lower())
        
        if selected_tool:
            # Wywołanie narzędzia
            tool_msg = selected_tool.invoke(tool_call)
            
            # Wydruk dla widoczności (jak w przykładzie)
            print(f'{tool_call["name"]} {tool_call["args"]} -> {tool_msg.content}')
            
            # Dodanie odpowiedzi narzędzia do historii
            messages.append(tool_msg)

    # --- 6. Finalna odpowiedź po wykonaniu narzędzi ---
    final_response = llm_with_tools.invoke(messages)
    print("\nFinal Response:")
    print(final_response.content)


multiply {'x': 393, 'y': 12.25} -> 4814.25
add {'x': 11, 'y': 49} -> 60.0

Final Response:
393 * 12.25 equals 4814.25, and 11 + 49 equals 60.


API TOOLS

In [17]:
import os
from langchain_openai import ChatOpenAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage, ToolMessage

# --- 1. Konfiguracja narzędzia Wikipedia ---
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


llm = ChatOpenAI(
    model=FREE_MODELS["nemotron_nano"],  # Wybieramy model z Twojej listy
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

# Podpięcie narzędzia (bind_tools)
llm_with_tools = llm.bind_tools([wiki_tool])

# --- 3. Przygotowanie zapytania ---
query = "What was the most impressive thing about XTB?"
messages = [HumanMessage(query)]

# --- 4. Pierwsze wywołanie (LLM decyduje o użyciu narzędzia) ---
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

# --- 5. Obsługa wywołań narzędzi ---
if hasattr(ai_msg, "tool_calls") and ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        # Wywołanie narzędzia (LangChain 0.3.7: .invoke() na tool_call)
        tool_msg = wiki_tool.invoke(tool_call)
        
        # Wydruk szczegółów (jak w oryginale)
        print(f"Tool: {tool_msg.name}")
        print(f"Args: {tool_call['args']}")
        print(f"Content: {tool_msg.content[:200]}...")  # Skróć długie treści
        
        messages.append(tool_msg)

# --- 6. Finalna odpowiedź ---
final_response = llm_with_tools.invoke(messages)
print("\nFinal Response:")
print(final_response.content)


Tool: wikipedia
Args: {'query': 'XTB'}
Content: Page: XTB S.A.
Summary: XTB S.A. is a Warsaw-based brokerage firm that provides products, services, and technology solutions for trading various financial instruments, including foreign exchange (fore...

Final Response:
The most impressive aspectof XTB, based on the available information, is that it is a **Warsaw-based brokerage firm founded in 2002** by Jakub Zabłocki, which has grown to become a significant player in the global financial markets, offering services for trading forex, CFDs, and other financial instruments. Its longevity and market presence highlight its reputation and innovation in the industry. For more specific details about its achievements or rankings, further research into its full Wikipedia page or official reports would be needed.
